# Curtailment model — challenger vs champion (spec 18)

Runs only after the drift check retrained a `challenger` (trained on everything before the monitor
window, its Y tuned on the window). Scores the window's labeled plant-hours with both versions and
promotes the challenger to `champion` only if, **for every technology**, it keeps flag precision ≥ 0.8
and has a lower capacity-factor MAE than the champion (`curtailment_lib.beats`). Otherwise the
champion stays. Both results and the decision are appended to `ampera_gold.curtailment_model_monitor`;
`promoted` is published so the job re-scores only when the champion changed.

In [ ]:
%pip install -q "mlflow>=3.1" scikit-learn

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
import os
import sys
from datetime import date, datetime, timezone

import mlflow
from mlflow import MlflowClient
from pyspark.sql import functions as F

sys.path.append(os.path.abspath("../src"))
import curtailment_lib as C  # noqa: E402  beats (tests/test_curtailment.py)
import curtailment_monitor as M  # noqa: E402

dbutils.widgets.text("catalog", "ampera_catalog")  # noqa: F821
dbutils.widgets.text("window_from", "")  # noqa: F821
CAT = dbutils.widgets.get("catalog")  # noqa: F821
MODEL = f"{CAT}.ampera_gold.curtailment_expected_cf"
MON = f"{CAT}.ampera_gold.curtailment_model_monitor"
mlflow.set_registry_uri("databricks-uc")

feat = spark.table(f"{CAT}.ampera_gold.curtailment_features_hourly")  # noqa: F821
w_from = date.fromisoformat(dbutils.widgets.get("window_from"))  # noqa: F821
w_to = M.window(feat)[1]
win = feat.filter(F.col("gen_date").between(w_from, w_to) & F.col("restrito_ons").isNotNull())

In [ ]:
res, ver = {}, {}
for alias in ("champion", "challenger"):
    scored, ver[alias] = M.score(spark, win, MODEL, alias)  # noqa: F821
    res[alias] = M.metrics(scored)
    print(f"{alias} v{ver[alias]}: {res[alias]}")

wins = {f: C.beats(res["challenger"][f], res["champion"][f]) for f in res["champion"]}
promote = all(wins.values()) and ver["challenger"] != ver["champion"]
decision = (f"promoted v{ver['challenger']} over v{ver['champion']}" if promote
            else f"kept v{ver['champion']}: challenger v{ver['challenger']} wins {wins}")
c = MlflowClient()
if promote:
    c.set_registered_model_alias(MODEL, "champion", ver["challenger"])
    c.set_model_version_tag(MODEL, ver["challenger"], "promoted_by", f"drift check, window {w_from}..{w_to}")
c.delete_registered_model_alias(MODEL, "challenger")
print(decision)

now = datetime.now(timezone.utc)
rows = [(now, alias, ver[alias], f, w_from, w_to, x["precision"], x["recall"], x["mae_model"], x["mae_ons"],
         x["flagged"], x["labeled_hours"], None, decision)
        for alias in ("champion", "challenger") for f, x in sorted(res[alias].items())]
spark.createDataFrame(rows, spark.table(MON).schema).write.mode("append").saveAsTable(MON)  # noqa: F821
dbutils.jobs.taskValues.set("promoted", "true" if promote else "false")  # noqa: F821

2026/10/27 23:10:48 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.
2026/10/27 23:10:48 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
champion v2: {'SOL': {'precision': 0.8516, 'recall': 0.8319, 'mae_model': 0.0119, 'mae_ons': 0.0134, 'flagged': 417513, 'labeled_hours': 1185312}, 'EOL': {'precision': 0.8609, 'recall': 0.5032, 'mae_model': 0.0944, 'mae_ons': 0.1045, 'flagged': 693424, 'labeled_hours': 2296776}}
2026/10/27 23:12:19 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We 